## 1. Install packages

Run this cell, then restart the Colab runtime once before continuing.

In [ ]:
!pip install -q -U \
  langchain==0.3.27 \
  langchain-core==0.3.75 \
  langchain-community==0.3.29 \
  langchain-groq \
  langchain-qdrant \
  qdrant-client \
  fastembed \
  llama-parse \
  gdown

## 2. Imports and API keys

In [ ]:
import os
import getpass
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_groq import ChatGroq
from llama_parse import LlamaParse

In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass.getpass("Enter your LlamaCloud API key: ")

## 3. Download course files

Replace the folder URL with your own shared Google Drive folder if needed.

In [ ]:
!rm -rf /content/chatbot
!gdown --folder "https://drive.google.com/drive/folders/15_e22ZnN3jH1Kx937CyRGoU1_XlNBVlg" -O /content/chatbot


In [ ]:
!find /content/chatbot -maxdepth 2 -type f | head -20

## 4. Load / parse documents

For `.txt` transcripts, we read them directly. For PDFs, DOCX, PPTX, etc., we use LlamaParse.

In [ ]:
parser = LlamaParse(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    result_type="markdown"
)

raw_docs = []
folder = Path("/content/chatbot")

for filepath in folder.rglob("*"):
    if not filepath.is_file():
        continue

    if filepath.suffix.lower() == ".txt":
        text = filepath.read_text(encoding="utf-8", errors="ignore")
        if text.strip():
            raw_docs.append(Document(page_content=text, metadata={"source": str(filepath)}))
    else:
        parsed = await parser.aload_data(str(filepath))
        for doc in parsed:
            if doc.text.strip():
                raw_docs.append(Document(page_content=doc.text, metadata={"source": str(filepath)}))

print("Raw documents loaded:", len(raw_docs))
print(raw_docs[0].page_content[:500])

## 5. Chunk documents

Chunking makes retrieval more precise. This step is optional if your documents are already small, but it is useful for long lecture transcripts.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(raw_docs)

print("Chunks created:", len(docs))
print(docs[0].page_content[:500])

## 6. Create embeddings and Qdrant vector store

In [ ]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

qdrant = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="course_documents",
)

retriever = qdrant.as_retriever(search_kwargs={"k": 5})

## 7. Create the LLM

In [ ]:
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile"
)


## 8. Ask questions over your course documents

This replaces old `RetrievalQA` and avoids the FlashRank compatibility problems.

In [ ]:
def ask_course_bot(question, k=5):
    relevant_docs = retriever.invoke(question)

    context = "\n\n---\n\n".join(
        doc.page_content for doc in relevant_docs[:k]
    )

    prompt = f"""
You are a helpful course assistant. Use only the context below to answer the question.
If the answer is not in the context, say you do not know based on the provided course materials.

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

    response = llm.invoke(prompt)
    return response.content, relevant_docs


answer, sources = ask_course_bot(
    "What does the lecture say about fraud?"
)

print(answer)

## 9. Interactive question box

In [ ]:
question = input("Enter your question: ")
answer, sources = ask_course_bot(question)
print(answer)

## Optional: inspect retrieved source chunks

In [ ]:
results = qdrant.similarity_search_with_score(question, k=5)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"Score: {score}")
    print(f"Source: {doc.metadata.get('source')}")
    print(doc.page_content[:300])
    print("="*80)

In [ ]:
import gradio as gr

def chatbot(question):
    answer, docs = ask_course_bot(question)

    source_text = "\n\n".join(
        [doc.metadata.get("source", "Unknown")
         for doc in docs]
    )

    return answer, source_text

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Retrieved Sources")
    ],
    title="AIS Course RAG Chatbot"
)

demo.launch()